# Match-stats EDA

What factors influence wins vs. losses for each tracked player.
Source: `Bot/db/database.sqlite` → `match_stats` joined to `league_players`.

Refresh data first with `scripts\sync-db.ps1`.
All plotting helpers live in `Bot/utils/match_analysis.py` (shared with the Discord cog).


In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'Bot'))

import pandas as pd
from utils import match_analysis as analysis

pd.set_option('display.max_rows', 40)
pd.set_option('display.width', 200)


## Load + summary


In [ ]:
df = analysis.load_matches()
print(f'{len(df):,} rows  /  {df["player"].nunique()} players  /  {df["game_start"].min().date()} → {df["game_start"].max().date()}')
df.head()


In [ ]:
summary = analysis.summary_table(df)
summary


## 1. Lifetime progression — do you get better with reps?

Rolling win rate vs. cumulative game number. Up-slope = improving, down-slope = declining. The aggregate panel labels each player with their linear-fit slope in pp/game.


In [ ]:
analysis.plot_player_progression(df, min_games=100);


In [ ]:
analysis.plot_cumulative_winrate(df);


## 2. KDA + game duration

Does carrying actually pay off, and do short stomps swing harder than long scaling games?


In [ ]:
analysis.plot_kda_vs_outcome(df);


In [ ]:
analysis.plot_duration_vs_outcome(df);


## 3. Champions — winners vs losers

Left panel: highest-WR champs (≥ min_games each). Right panel: lowest-WR. Annotated with sample count so a 70%-on-3-games doesn't masquerade as a winner.


In [ ]:
analysis.plot_champion_winrate(df, min_games=20, top=12);


In [ ]:
analysis.plot_champion_learning_curve(df, top=5, window=5);


## 4. Time-of-day + day-of-week

Tilt hours, prime hours, weekend warrior patterns. The heatmap is the useful one — 1-D charts hide things like 'only Friday-evening is bad'.


In [ ]:
analysis.plot_hour_of_day(df);


In [ ]:
analysis.plot_day_of_week(df);


In [ ]:
analysis.plot_hour_dow_heatmap(df);


## 5. Recent form & momentum

Tilt check: does losing-streak length predict the next game's outcome? Back-to-back queue vs fresh session: does the gap-since-last-game affect winrate?


In [ ]:
analysis.plot_streak_recovery(df);


In [ ]:
analysis.plot_time_since_prev(df);


---
## Per-player drill-down

Change `PLAYER` below and re-run the cells. Every chart accepts a `player=` kwarg.


In [ ]:
PLAYER = 'loukia'  # change me
print(summary[summary['player'] == PLAYER])


In [ ]:
analysis.plot_player_progression(df, player=PLAYER);


In [ ]:
analysis.plot_cumulative_winrate(df, player=PLAYER);


In [ ]:
analysis.plot_kda_vs_outcome(df, player=PLAYER);


In [ ]:
analysis.plot_duration_vs_outcome(df, player=PLAYER);


In [ ]:
analysis.plot_champion_winrate(df, player=PLAYER, min_games=5, top=10);


In [ ]:
analysis.plot_champion_learning_curve(df, player=PLAYER, top=5, window=5);


In [ ]:
analysis.plot_hour_of_day(df, player=PLAYER);


In [ ]:
analysis.plot_day_of_week(df, player=PLAYER);


In [ ]:
analysis.plot_hour_dow_heatmap(df, player=PLAYER);


In [ ]:
analysis.plot_streak_recovery(df, player=PLAYER);


In [ ]:
analysis.plot_time_since_prev(df, player=PLAYER);


---
## Render every chart for every player as PNG

Equivalent to `python notebooks\run_analysis.py` — useful for batch sharing.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'run_analysis.py'], cwd='.', check=True)
